In [2]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
import lightgbm as lgb
import torch.nn as nn
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [14]:
def calculate_interval_score(y_true, lower, upper, alpha):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2.0 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2.0 / alpha) * (y_true - upper), 0)
    return np.mean(width + penalty_lower + penalty_upper)



#POINT MLP ARCHITECTURE
class PointMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(1536, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, 1) # Outputs a SINGLE price
        )
    def forward(self, x): return self.head(x)

In [15]:
def run_point_prediction_baselines(split_dir="split_embeddings", epochs=15):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running Point Baselines on {device}...\n")
    
    category_t_map = {
        "BIKES": 0.94, "BOOKS": 0.97, "CARS": 0.95, "CYCLE": 0.96,
        "FLAT": 0.94, "FRIDGES": 0.94, "GAMES": 0.94, "GAMESENTERTAINMENT": 0.97,
        "LAPTOP": 0.95, "MOBILE": 0.95, "PHONES": 0.98, "PRINTER": 0.95,
        "TV": 0.96, "WASHINGMACHINE": 0.94
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val
        
        # Calculate dynamic percentiles for the residuals
        lower_percentile = (alpha / 2.0) * 100
        upper_percentile = (1.0 - (alpha / 2.0)) * 100
        
        print(f"\nCATEGORY: {cat_name} ")
        print(f"\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        raw_test_data = torch.load(test_path, weights_only=False)
        
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        X_full_train = raw_train_data['embeddings'].numpy()
        y_full_train = ((raw_train_data['log_prices'] - cat_mu) / cat_sigma).numpy()
        X_test = raw_test_data['embeddings'].numpy()
        y_test_real = np.exp(raw_test_data['log_prices'].numpy())
        
        if len(X_full_train) < 30:
            continue
            
        # Split Train into Train (80%) and Calibration (20%) for honest residual errors
        X_train, X_calib, y_train, y_calib = train_test_split(
            X_full_train, y_full_train, test_size=0.2, random_state=54
        )
        
        # MODEL 1: RIDGE REGRESSION
        model_linear = Ridge(alpha=1.0)
        model_linear.fit(X_train, y_train)
        
        calib_preds = model_linear.predict(X_calib)
        residuals = y_calib - calib_preds
        e_low = np.percentile(residuals, lower_percentile)
        e_high = np.percentile(residuals, upper_percentile)
        
        test_preds = model_linear.predict(X_test)
        test_preds_real = np.exp(test_preds * cat_sigma + cat_mu)
        rmse_point_lin = np.sqrt(np.mean((y_test_real - test_preds_real) ** 2))
        l_real_lin = np.exp((test_preds + e_low) * cat_sigma + cat_mu)
        u_real_lin = np.exp((test_preds + e_high) * cat_sigma + cat_mu)
        
        picp_lin = np.mean((y_test_real >= l_real_lin) & (y_test_real <= u_real_lin))
        mpiw_lin = np.mean(u_real_lin - l_real_lin)
        is_lin = calculate_interval_score(y_test_real, l_real_lin, u_real_lin, alpha)
        
        print(f"\tRIDGE REGRESSION ")
        print(f"POINT RMSE: ₹{rmse_point_lin:,.0f}")
        print(f"PICP: {picp_lin:.4f} | MPIW: ₹{mpiw_lin:,.0f} | IS: {is_lin:,.0f} ")


        # MODEL 2: LIGHTGBM POINT PREDICTOR
        model_lgb = lgb.LGBMRegressor(n_estimators=100, random_state=54, verbose=-1)
        model_lgb.fit(X_train, y_train)
        
        calib_preds = model_lgb.predict(X_calib)
        residuals = y_calib - calib_preds
        e_low = np.percentile(residuals, lower_percentile)
        e_high = np.percentile(residuals, upper_percentile)
        
        test_preds = model_lgb.predict(X_test)
        test_preds_real = np.exp(test_preds * cat_sigma + cat_mu)
        rmse_point_lgb = np.sqrt(np.mean((y_test_real - test_preds_real) ** 2))
        l_real_lgb = np.exp((test_preds + e_low) * cat_sigma + cat_mu)
        u_real_lgb = np.exp((test_preds + e_high) * cat_sigma + cat_mu)
        
        picp_lgb = np.mean((y_test_real >= l_real_lgb) & (y_test_real <= u_real_lgb))
        mpiw_lgb = np.mean(u_real_lgb - l_real_lgb)
        is_lgb = calculate_interval_score(y_test_real, l_real_lgb, u_real_lgb, alpha)
        
        print(f"\tLIGHTGBM (POINT) ")
        print(f"POINT RMSE: ₹{rmse_point_lgb:,.0f}")
        print(f"PICP: {picp_lgb:.4f} | MPIW: ₹{mpiw_lgb:,.0f} | IS: {is_lgb:,.0f}")

        # MODEL 3: DEEP MLP POINT PREDICTOR
        model_mlp = PointMLP().to(device)
        optimizer = AdamW(model_mlp.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        
        X_tr_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
        
        model_mlp.train()
        for _ in range(epochs):
            optimizer.zero_grad()
            out = model_mlp(X_tr_t)
            loss = criterion(out, y_tr_t)
            loss.backward()
            optimizer.step()
            
        model_mlp.eval()
        with torch.no_grad():
            X_calib_t = torch.tensor(X_calib, dtype=torch.float32).to(device)
            calib_preds = model_mlp(X_calib_t).cpu().numpy().flatten()
            
        residuals = y_calib - calib_preds
        e_low = np.percentile(residuals, lower_percentile)
        e_high = np.percentile(residuals, upper_percentile)
        
        with torch.no_grad():
            X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
            test_preds = model_mlp(X_test_t).cpu().numpy().flatten()
        
        test_preds_real = np.exp(test_preds * cat_sigma + cat_mu)
        rmse_point_mlp = np.sqrt(np.mean((y_test_real - test_preds_real) ** 2))
        l_real_mlp = np.exp((test_preds + e_low) * cat_sigma + cat_mu)
        u_real_mlp = np.exp((test_preds + e_high) * cat_sigma + cat_mu)
        
        picp_mlp = np.mean((y_test_real >= l_real_mlp) & (y_test_real <= u_real_mlp))
        mpiw_mlp = np.mean(u_real_mlp - l_real_mlp)
        is_mlp = calculate_interval_score(y_test_real, l_real_mlp, u_real_mlp, alpha)
        
        print(f"\tDEEP MLP (POINT)")
        print(f"POINT RMSE: ₹{rmse_point_mlp:,.0f}")
        print(f"PICP: {picp_mlp:.4f} | MPIW: ₹{mpiw_mlp:,.0f} | IS: {is_mlp:,.0f}")



In [17]:
torch.manual_seed(54)
run_point_prediction_baselines(epochs=5)

Running Point Baselines on cuda...


CATEGORY: BIKES 

--------------------------------------------------
	RIDGE REGRESSION 
POINT RMSE: ₹95,600
PICP: 0.9773 | MPIW: ₹234,503 | IS: 535,021 
	LIGHTGBM (POINT) 
POINT RMSE: ₹93,465
PICP: 0.9773 | MPIW: ₹227,471 | IS: 567,464
	DEEP MLP (POINT)
POINT RMSE: ₹102,142
PICP: 0.9545 | MPIW: ₹215,855 | IS: 519,576

CATEGORY: BOOKS 

--------------------------------------------------
	RIDGE REGRESSION 
POINT RMSE: ₹3,783
PICP: 0.9130 | MPIW: ₹11,448 | IS: 32,309 
	LIGHTGBM (POINT) 
POINT RMSE: ₹3,852
PICP: 0.9783 | MPIW: ₹16,072 | IS: 28,113
	DEEP MLP (POINT)
POINT RMSE: ₹3,914
PICP: 0.9348 | MPIW: ₹9,915 | IS: 28,715

CATEGORY: CARS 

--------------------------------------------------
	RIDGE REGRESSION 
POINT RMSE: ₹581,092
PICP: 0.8065 | MPIW: ₹1,897,206 | IS: 3,586,660 
	LIGHTGBM (POINT) 
POINT RMSE: ₹466,025
PICP: 0.7419 | MPIW: ₹1,626,020 | IS: 2,498,855
	DEEP MLP (POINT)
POINT RMSE: ₹632,138
PICP: 0.8065 | MPIW: ₹3,440,548 | IS: 3,944,512

C